In [ ]:
repository_filter: list[str] = []
min_wmc: str = "5"
data_file_gaps: str = "../samples/test_gaps.csv"

In [ ]:
from code_data_science import data_table as dt
from moderne_visualizations_misc.reusable import quality_utils as qu
import plotly.graph_objects as go
import pandas as pd
import warnings

warnings.simplefilter("ignore")

wmc_min = int(min_wmc)

class_df = dt.read_csv("../samples/class_quality_metrics.csv")
class_df = qu.filter_repos(class_df, repository_filter)
class_df = qu.add_repo_short(class_df)
class_df = qu.add_class_short(class_df)

gaps_df = pd.read_csv(data_file_gaps, on_bad_lines="skip")
gaps_df = qu.filter_repos(gaps_df, repository_filter)

In [ ]:
if len(class_df) == 0 or len(gaps_df) == 0:
    fig = qu.empty_figure("No data available for class metrics or test gaps.")
else:
    # Aggregate gaps by className + repositoryPath
    gap_agg = (
        gaps_df.groupby(["repositoryPath", "className"])
        .agg(
            untested_methods=("methodName", "count"),
            total_risk=("riskScore", "sum"),
        )
        .reset_index()
    )

    # Merge with class metrics
    merged = class_df.merge(
        gap_agg,
        on=["repositoryPath", "className"],
        how="inner",
    )

    # Filter by minimum WMC
    merged = merged[merged["wmc"] >= wmc_min]

    if len(merged) == 0:
        fig = qu.empty_figure(
            f"No classes with WMC >= {wmc_min} and test gaps found."
        )
    else:
        # Scale bubble sizes
        risk_max = merged["total_risk"].max()
        risk_scale = merged["total_risk"] / risk_max if risk_max > 0 else 1
        sizes = 10 + 50 * risk_scale

        # Median lines for quadrant effect
        wmc_median = merged["wmc"].median()
        gap_median = merged["untested_methods"].median()

        fig = go.Figure()

        fig.add_trace(
            go.Scatter(
                x=merged["wmc"],
                y=merged["untested_methods"],
                mode="markers",
                marker=dict(
                    size=sizes,
                    color=merged["maintainabilityIndex"],
                    colorscale=qu.HEALTH_COLORSCALE,
                    cmin=0,
                    cmax=100,
                    colorbar=dict(title="Maintainability<br>Index"),
                    line=dict(width=0.5, color="#ddd"),
                ),
                text=merged["className"],
                customdata=merged[
                    ["repoShort", "total_risk", "cbo", "lcom4"]
                ].values,
                hovertemplate=(
                    "<b>%{text}</b><br>"
                    "Repo: %{customdata[0]}<br>"
                    "WMC: %{x}<br>"
                    "Untested methods: %{y}<br>"
                    "Total risk: %{customdata[1]}<br>"
                    "CBO: %{customdata[2]}<br>"
                    "LCOM4: %{customdata[3]}<br>"
                    "Maintainability: %{marker.color:.0f}"
                    "<extra></extra>"
                ),
            )
        )

        # Median lines for quadrant effect
        fig.add_hline(
            y=gap_median, line_dash="dash",
            line_color="#888", opacity=0.5,
            annotation_text=f"Median gaps ({gap_median:.0f})",
        )
        fig.add_vline(
            x=wmc_median, line_dash="dash",
            line_color="#888", opacity=0.5,
            annotation_text=f"Median WMC ({wmc_median:.0f})",
        )

        fig.update_layout(
            title="Complexity vs Test Gaps",
            xaxis_title="WMC (Weighted Methods per Class)",
            yaxis_title="Untested Methods",
            height=700,
            width=900,
            plot_bgcolor="white",
            showlegend=False,
            margin=dict(l=60, r=60, t=60, b=60),
        )
fig.show()